In [3]:
import pandas as pd
import numpy as np

from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from xailib.data_loaders.dataframe_loader import prepare_dataframe

from xailib.explainers.lime_explainer import LimeXAITabularExplainer
from xailib.explainers.lore_explainer import LoreTabularExplainer
from xailib.explainers.shap_explainer_tab import ShapXAITabularExplainer

from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

In [4]:
import pickle
import altair as alt

In [5]:
import os

In [6]:
path = os.getcwd()
print(path)

/Users/eleonoracappuccio/XAI-Lib_WD/rules_VIZ


# Learning and explaining German Credit Dataset + visualize rules

## Loading and preparation of data

We start by reading from a CSV file the dataset to analyze. The table is loaded by means of the ```DataFrame``` class from the ```pandas``` library.

Among all the attributes of the table, we select the ```class_field``` column that contains the observed class for the corresponding row.

In [7]:
source_file = '/Users/eleonoracappuccio/XAI-Lib_WD/datasets/german_credit.csv' 
class_field = 'default'
# Load and transform dataset 
df = pd.read_csv(source_file, skipinitialspace=True, na_values='?', keep_default_na=True)

In [8]:
df

,default,account_check_status,duration_in_month,credit_history,purpose,credit_amount,savings,present_emp_since,installment_as_income_perc,personal_status_sex,...,present_res_since,property,age,other_installment_plans,housing,credits_this_bank,job,people_under_maintenance,telephone,foreign_worker
0,0,< 0 DM,6,critical account/ other credits existing (not ...,domestic appliances,1169,unknown/ no savings account,.. >= 7 years,4,male : single,...,4,real estate,67,none,own,2,skilled employee / official,1,"yes, registered under the customers name",yes
1,1,0 <= ... < 200 DM,48,existing credits paid back duly till now,domestic appliances,5951,... < 100 DM,1 <= ... < 4 years,2,female : divorced/separated/married,...,2,real estate,22,none,own,1,skilled employee / official,1,none,yes
2,0,no checking account,12,critical account/ other credits existing (not ...,(vacation - does not exist?),2096,... < 100 DM,4 <= ... < 7 years,2,male : single,...,3,real estate,49,none,own,1,unskilled - resident,2,none,yes
3,0,< 0 DM,42,existing credits paid back duly till now,radio/television,7882,... < 100 DM,4 <= ... < 7 years,2,male : single,...,4,if not A121 : building society savings agreeme...,45,none,for free,1,skilled employee / official,2,none,yes
4,1,< 0 DM,24,delay in paying off in the past,car (new),4870,... < 100 DM,1 <= ... < 4 years,3,male : single,...,4,unknown / no property,53,none,for free,2,skilled employee / official,2,none,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,no checking account,12,existing credits paid back duly till now,radio/television,1736,... < 100 DM,4 <= ... < 7 years,3,female : divorced/separated/married,...,4,real estate,31,none,own,1,unskilled - resident,1,none,yes
996,0,< 0 DM,30,existing credits paid back duly till now,car (used),3857,... < 100 DM,1 <= ... < 4 years,4,male : divorced/separated,...,4,if not A121 : building society savings agreeme...,40,none,own,1,management/ self-employed/ highly qualified em...,1,"yes, registered under the customers name",yes
997,0,no checking account,12,existing credits paid back duly till now,domestic appliances,804,... < 100 DM,.. >= 7 years,4,male : single,...,4,"if not A121/A122 : car or other, not in attrib...",38,none,own,1,skilled employee / official,1,none,yes
998,1,< 0 DM,45,existing credits paid back duly till now,domestic appliances,1845,... < 100 DM,1 <= ... < 4 years,4,male : single,...,4,unknown / no property,23,none,for free,1,skilled employee / official,1,"yes, registered under the customers name",yes


After the data is loaded in memory, we need to extract metadata information to automatically handle the content withint the table.

The method ```prepare_dataframe``` scans the table and extract the following information:
 * ```df```: is a trasformed version of the original dataframe, where discrete attributes are transformed into numerical attributes by using one hot encoding strategy;
 * ```feature_names```: is a list containing the names of the features after the transformation;
 * ```class_values```: the list of all the possible values for the ```class_field``` column;
 * ```numeric_columns```: a list of the original features that contain numeric (i.e. continuous) values;
 * ```rdf```: the original dataframe, before the transformation;
 * ```real_feature_names```: the list of the features of the dataframe before the transformation;
 * ```features_map```: it is a dictionary pointing each feature to the original one before the transformation.

In [6]:
df, feature_names, class_values, numeric_columns, rdf, real_feature_names, features_map = prepare_dataframe(df, class_field)

### Learning a Random Forest classfier

We train a RF classifier by using the ```sklearn``` library. We start by splitting the dataset into a train and test subsets. 

In [7]:
test_size = 0.3
random_state = 42
X_train, X_test, Y_train, Y_test = train_test_split(df[feature_names], df[class_field],
                                                        test_size=test_size,
                                                        random_state=random_state,
                                                        stratify=df[class_field])


Then we train the model on the training set. 
Once the model has been learned, we use a wrapper class to get access to the model for ```XAI lib```

In [8]:
bb = RandomForestClassifier(n_estimators=20, random_state=random_state)
bb.fit(X_train.values, Y_train.values)
bbox = sklearn_classifier_wrapper(bb)

Select a new instance to be classfied by the model and print the predicted class.

In [9]:
inst = X_train.iloc[0].values
print('Instance ',inst)
print('True class ',Y_train.iloc[8])
print('Predicted class ',bb.predict(inst.reshape(1, -1)))

Instance  [  12 1295    3    1   25    1    1    1    0    0    0    0    0    0
    1    0    0    0    1    0    0    0    0    0    0    0    0    1
    0    0    0    0    1    0    0    0    1    0    0    0    0    0
    1    0    1    0    0    0    1    0    0    0    1    0    1    0
    0    1    0    0    1]
True class  0
Predicted class  [1]


In [10]:
real_inst = inst
print(real_inst)

[  12 1295    3    1   25    1    1    1    0    0    0    0    0    0
    1    0    0    0    1    0    0    0    0    0    0    0    0    1
    0    0    0    0    1    0    0    0    1    0    0    0    0    0
    1    0    1    0    0    0    1    0    0    0    1    0    1    0
    0    1    0    0    1]


## Explaining the prediction
We use the explanators of ```XAI lib``` to provide an explantion for the classified instance ```inst```.
Every explainer of ```XAI lib``` takes in input the blackbox to be explained with the corresponding feature names, and a configuration object to initialize the explainer.

### SHAP explainer

In [11]:
explainer = ShapXAITabularExplainer(bbox, feature_names)
config = {'explainer' : 'tree', 'X_train' : X_train.iloc[0:100].values}
explainer.fit(config)

In [12]:
exp = explainer.explain(inst)

In [13]:
exp.plot_features_importance()

alt.VConcatChart(...)

### LORE explainer (non funziona)

In [27]:
type(real_inst)
#non so se è giusto ma ora appiattisco inst 
inst_piatto = real_inst.reshape(-1)
print(inst_piatto)


[  12 1295    3    1   25    1    1    1    0    0    0    0    0    0
    1    0    0    0    1    0    0    0    0    0    0    0    0    1
    0    0    0    0    1    0    0    0    1    0    0    0    0    0
    1    0    1    0    0    0    1    0    0    0    1    0    1    0
    0    1    0    0    1]


In [28]:
explainer = LoreTabularExplainer(bbox)
config = {'neigh_type':'rndgen', 'size':1000, 'ocr':0.1, 'ngen':10}
explainer.fit(df, class_field, config)
exp = explainer.explain(inst_piatto)
print(exp)

ValueError: Input vector should be 1-D.

In [ ]:
exp.plotRules()

In [17]:
exp.plotCounterfactualRules()

AttributeError: 'ShapXAITabularExplanation' object has no attribute 'plotCounterfactualRules'

### LIME explainer

In [31]:
limeExplainer = LimeXAITabularExplainer(bbox)
config = {'feature_selection': 'lasso_path'}
limeExplainer.fit(df, class_field, config)
lime_exp = limeExplainer.explain(inst)
print(lime_exp.exp.as_list())

[('duration_in_month', 0.04304677882516805), ('account_check_status=no checking account', -0.03498281206894297), ('account_check_status=< 0 DM', 0.03147808748357848), ('credit_history=critical account/ other credits existing (not at this bank)', -0.024149716261148604), ('savings=... < 100 DM', 0.023021018797555186), ('present_emp_since=... < 1 year ', 0.020005235354574446), ('age', -0.01809277718722266), ('property=real estate', -0.017723229453490596), ('savings=unknown/ no savings account', -0.017579308403499083), ('account_check_status=0 <= ... < 200 DM', 0.015627770595178076)]


In [32]:
# limeExplainer.plot_lime_values(lime_exp.as_list(), 5, 10)
lime_exp.plot_features_importance()

alt.VConcatChart(...)

## Learning a different model

### Learning a Logistic Regressor

We train a Logistic Regression by using the ```sklearn``` library. We transform the dataset by using a ```Scaler``` to normalize all the attributes.


In [33]:
scaler = preprocessing.StandardScaler().fit(X_train)
X_scaled = scaler.transform(X_train)

bb = LogisticRegression(C=1, penalty='l2')
bb.fit(X_scaled, Y_train.values)
# pass the model to the wrapper to use it in the XAI lib
bbox = sklearn_classifier_wrapper(bb)

In [34]:
# select a record to explain
inst = X_scaled[182]
print('Instance ',inst)
print('Predicted class ',bb.predict(inst.reshape(1, -1)))

Instance  [ 2.27797454  3.35504085  0.94540357  1.07634233  0.04854891 -0.72456474
 -0.43411405  1.65027399 -0.61477862 -0.25898489 -0.80681063  4.17385345
 -0.6435382  -0.32533856 -1.03489416 -0.20412415 -0.22941573 -0.33068147
  1.75885396 -0.34899122 -0.60155441 -0.15294382 -0.09298136 -0.46852129
 -0.12038585 -0.08481889 -0.23623492 -1.21387736 -0.36174054 -0.24943031
  2.15526362 -0.59715086 -0.45485883 -0.73610476 -0.43875307  4.23307441
 -0.65242771 -0.23958675 -0.32533856  0.90192655  4.72581563 -0.2259448
 -3.15238005 -0.54212562 -0.70181003 -0.63024248  2.30354212 -0.40586384
  0.49329429 -0.23958675  2.88675135 -1.59227935 -0.46170508  2.46388049
 -1.33747696 -0.13206764 -0.5        -1.21387736  1.21387736 -0.20412415
  0.20412415]
Predicted class  [1]


In [35]:
X_scaled

array([[-0.7335121 , -0.71300074,  0.0547138 , ..., -0.82380645,
        -0.20412415,  0.20412415],
       [-0.23159766, -0.61086948,  0.0547138 , ..., -0.82380645,
        -0.20412415,  0.20412415],
       [-0.23159766,  0.3606869 , -0.83597597, ..., -0.82380645,
        -0.20412415,  0.20412415],
       ...,
       [-0.23159766, -0.25658997,  0.0547138 , ...,  1.21387736,
        -0.20412415,  0.20412415],
       [-0.23159766,  1.97159246, -1.72666575, ...,  1.21387736,
        -0.20412415,  0.20412415],
       [-0.48255488, -0.06429887, -0.83597597, ..., -0.82380645,
        -0.20412415,  0.20412415]])

## Explaining the prediction
We use the same explainators as for the previous model. In this case, a few adjustments are necessary for the initialization of the explanators. For example, SHAP needs a specific configuration for the linear model we are using.
### SHAP Explainer

In [36]:
explainer = ShapXAITabularExplainer(bbox, feature_names)
config = {'explainer' : 'linear', 'X_train' : X_scaled[0:100], 'feature_pert' : 'interventional'}
explainer.fit(config)

In [37]:
exp = explainer.explain(inst)
print(exp)

In [38]:
exp.plot_features_importance()

alt.VConcatChart(...)

### LORE explainer

In [39]:
explainer = LoreTabularExplainer(bbox)
config = {'neigh_type':'geneticp', 'size':1000, 'ocr':0.1, 'ngen':10}
explainer.fit(df, class_field, config)
exp = explainer.explain(inst)
print(exp)

In [40]:
exp.plotRules()

In [41]:
exp.plotCounterfactualRules()

### LIME explainer

In [42]:
limeExplainer = LimeXAITabularExplainer(bbox)
config = {'feature_selection': 'lasso_path'}
limeExplainer.fit(df, class_field, config)
lime_exp = limeExplainer.explain(inst)
print(lime_exp.exp.as_list())

[('other_debtors=co-applicant', -1.209948448325506e-09), ('credit_history=all credits at this bank paid back duly', -9.80729285586005e-10), ('present_emp_since=unemployed', -8.905493791597408e-10), ('other_debtors=none', 6.501490037885393e-10), ('housing=for free', -4.5100744231654407e-10), ('credit_amount', 3.4347399311566464e-10), ('job=management/ self-employed/ highly qualified employee/ officer', -3.0720840863891687e-10), ('property=unknown / no property', -3.0472604540923634e-10), ('savings=unknown/ no savings account', -3.0122747825365163e-10), ('savings=... < 100 DM', 2.405511062391578e-10), ('credits_this_bank', 2.371877264255113e-10), ('housing=own', 2.198182579019186e-10), ('property=real estate', 1.942527163166689e-10), ('account_check_status=0 <= ... < 200 DM', -1.8341777236604433e-10), ('foreign_worker=yes', 1.73514595276603e-10), ('purpose=car (new)', -1.6299442716105227e-10), ('account_check_status=no checking account', 1.6227117790107656e-10), ('people_under_maintenanc

In [43]:
lime_exp.plot_features_importance()

alt.VConcatChart(...)

In [44]:
rules =exp.expDict['rule']['premise']

In [45]:
rules

[{'att': 'age', 'op': '<=', 'thr': 20.726173400878906, 'is_continuous': True},
 {'att': 'credit_amount',
  'op': '>',
  'thr': -439.6443485021591,
  'is_continuous': True},
 {'att': 'purpose=retraining',
  'op': '<=',
  'thr': 0.11524588242173195,
  'is_continuous': True},
 {'att': 'duration_in_month',
  'op': '>',
  'thr': -1.9407005310058594,
  'is_continuous': True},
 {'att': 'purpose=furniture/equipment',
  'op': '<=',
  'thr': 0.18370826542377472,
  'is_continuous': True},
 {'att': 'foreign_worker=no',
  'op': '<=',
  'thr': 0.7168410122394562,
  'is_continuous': True},
 {'att': 'purpose=domestic appliances',
  'op': '<=',
  'thr': 1.015466570854187,
  'is_continuous': True},
 {'att': 'savings=.. >= 1000 DM ',
  'op': '<=',
  'thr': 0.7176859378814697,
  'is_continuous': True},
 {'att': 'purpose=(vacation - does not exist?)',
  'op': '<=',
  'thr': 0.4622504562139511,
  'is_continuous': True},
 {'att': 'credit_history=critical account/ other credits existing (not at this bank)',
 

In [46]:
for r in rules:
    print(r['att'])

age
credit_amount
purpose=retraining
duration_in_month
purpose=furniture/equipment
foreign_worker=no
purpose=domestic appliances
savings=.. >= 1000 DM 
purpose=(vacation - does not exist?)
credit_history=critical account/ other credits existing (not at this bank)
people_under_maintenance


In [47]:
df_range=pd.concat({'min':X_train.min(), 'max':X_train.max()},axis=1)

In [48]:
df_range=df_range.reset_index()

In [49]:
df_range

,index,min,max
0,duration_in_month,4,60
1,credit_amount,338,15945
2,installment_as_income_perc,1,4
3,present_res_since,1,4
4,age,19,75
...,...,...,...
56,job=unskilled - resident,0,1
57,telephone=none,0,1
58,"telephone=yes, registered under the customers ...",0,1
59,foreign_worker=no,0,1


In [50]:
rules

[{'att': 'age', 'op': '<=', 'thr': 20.726173400878906, 'is_continuous': True},
 {'att': 'credit_amount',
  'op': '>',
  'thr': -439.6443485021591,
  'is_continuous': True},
 {'att': 'purpose=retraining',
  'op': '<=',
  'thr': 0.11524588242173195,
  'is_continuous': True},
 {'att': 'duration_in_month',
  'op': '>',
  'thr': -1.9407005310058594,
  'is_continuous': True},
 {'att': 'purpose=furniture/equipment',
  'op': '<=',
  'thr': 0.18370826542377472,
  'is_continuous': True},
 {'att': 'foreign_worker=no',
  'op': '<=',
  'thr': 0.7168410122394562,
  'is_continuous': True},
 {'att': 'purpose=domestic appliances',
  'op': '<=',
  'thr': 1.015466570854187,
  'is_continuous': True},
 {'att': 'savings=.. >= 1000 DM ',
  'op': '<=',
  'thr': 0.7176859378814697,
  'is_continuous': True},
 {'att': 'purpose=(vacation - does not exist?)',
  'op': '<=',
  'thr': 0.4622504562139511,
  'is_continuous': True},
 {'att': 'credit_history=critical account/ other credits existing (not at this bank)',
 

In [51]:
df_rules = pd.DataFrame.from_records(rules)
df_rules

,att,op,thr,is_continuous
0,age,<=,20.726173,True
1,credit_amount,>,-439.644349,True
2,purpose=retraining,<=,0.115246,True
3,duration_in_month,>,-1.940701,True
4,purpose=furniture/equipment,<=,0.183708,True
5,foreign_worker=no,<=,0.716841,True
6,purpose=domestic appliances,<=,1.015467,True
7,savings=.. >= 1000 DM,<=,0.717686,True
8,purpose=(vacation - does not exist?),<=,0.462250,True
9,credit_history=critical account/ other credits...,<=,0.908596,True


In [52]:
df_viz = df_range.merge(df_rules,how='left',left_on='index',right_on='att')
df_viz = df_viz.drop('att', axis=1)
df_viz

,index,min,max,op,thr,is_continuous
0,duration_in_month,4,60,>,-1.940701,True
1,credit_amount,338,15945,>,-439.644349,True
2,installment_as_income_perc,1,4,NaN,NaN,NaN
3,present_res_since,1,4,NaN,NaN,NaN
4,age,19,75,<=,20.726173,True
...,...,...,...,...,...,...
56,job=unskilled - resident,0,1,NaN,NaN,NaN
57,telephone=none,0,1,NaN,NaN,NaN
58,"telephone=yes, registered under the customers ...",0,1,NaN,NaN,NaN
59,foreign_worker=no,0,1,<=,0.716841,True


In [53]:
df_viz['inst'] = real_inst.tolist()
df_viz

,index,min,max,op,thr,is_continuous,inst
0,duration_in_month,4,60,>,-1.940701,True,12
1,credit_amount,338,15945,>,-439.644349,True,1295
2,installment_as_income_perc,1,4,NaN,NaN,NaN,3
3,present_res_since,1,4,NaN,NaN,NaN,1
4,age,19,75,<=,20.726173,True,25
...,...,...,...,...,...,...,...
56,job=unskilled - resident,0,1,NaN,NaN,NaN,0
57,telephone=none,0,1,NaN,NaN,NaN,1
58,"telephone=yes, registered under the customers ...",0,1,NaN,NaN,NaN,0
59,foreign_worker=no,0,1,<=,0.716841,True,0


## Explaining the prediction
We use the explanators of ```XAI lib``` to provide an explantion for the classified instance ```inst```.
Every explainer of ```XAI lib``` takes in input the blackbox to be explained with the corresponding feature names, and a configuration object to initialize the explainer.

### SHAP explainer

In [54]:
explainer = ShapXAITabularExplainer(bbox, feature_names)
config = {'explainer' : 'tree', 'X_train' : X_train.iloc[0:100].values}
config

{'explainer': 'tree',
 'X_train': array([[  12, 1295,    3, ...,    0,    0,    1],
        [  18, 1568,    3, ...,    0,    0,    1],
        [  18, 4165,    2, ...,    0,    0,    1],
        ...,
        [  36, 6229,    4, ...,    1,    0,    1],
        [   6,  338,    4, ...,    0,    0,    1],
        [  12, 1185,    3, ...,    0,    0,    1]])}

In [55]:
explainer.fit(config)

InvalidModelError: Model type not yet supported by TreeExplainer: <class 'sklearn.linear_model._logistic.LogisticRegression'>

In [56]:
exp = explainer.explain(inst)
print(exp.exp)

AttributeError: 'NoneType' object has no attribute 'shap_values'

In [57]:
exp.plot_features_importance()

AttributeError: 'LoreTabularExplanation' object has no attribute 'plot_features_importance'

### LORE explainer

In [58]:
explainer = LoreTabularExplainer(bbox)
config = {'neigh_type':'geneticp', 'size':1000, 'ocr':0.1, 'ngen':10}
explainer.fit(df, class_field, config)
exp = explainer.explain(inst)
print(exp)

In [59]:
exp.plotRules()

In [47]:
exp.plotCounterfactualRules()

### LIME explainer

In [48]:
limeExplainer = LimeXAITabularExplainer(bbox)
config = {'feature_selection': 'lasso_path'}
limeExplainer.fit(df, class_field, config)
lime_exp = limeExplainer.explain(inst)
print(lime_exp.exp.as_list())

[('other_debtors=co-applicant', -1.209948448325506e-09), ('credit_history=all credits at this bank paid back duly', -9.80729285586005e-10), ('present_emp_since=unemployed', -8.905493791597408e-10), ('other_debtors=none', 6.501490037885393e-10), ('housing=for free', -4.5100744231654407e-10), ('credit_amount', 3.4347399311566464e-10), ('job=management/ self-employed/ highly qualified employee/ officer', -3.0720840863891687e-10), ('property=unknown / no property', -3.0472604540923634e-10), ('savings=unknown/ no savings account', -3.0122747825365163e-10), ('savings=... < 100 DM', 2.405511062391578e-10), ('credits_this_bank', 2.371877264255113e-10), ('housing=own', 2.198182579019186e-10), ('property=real estate', 1.942527163166689e-10), ('account_check_status=0 <= ... < 200 DM', -1.8341777236604433e-10), ('foreign_worker=yes', 1.73514595276603e-10), ('purpose=car (new)', -1.6299442716105227e-10), ('account_check_status=no checking account', 1.6227117790107656e-10), ('people_under_maintenanc

In [49]:
lime_exp.plot_features_importance()

alt.VConcatChart(...)

In [50]:
rules

[{'att': 'age', 'op': '<=', 'thr': 20.726173400878906, 'is_continuous': True},
 {'att': 'credit_amount',
  'op': '>',
  'thr': -439.6443485021591,
  'is_continuous': True},
 {'att': 'purpose=retraining',
  'op': '<=',
  'thr': 0.11524588242173195,
  'is_continuous': True},
 {'att': 'duration_in_month',
  'op': '>',
  'thr': -1.9407005310058594,
  'is_continuous': True},
 {'att': 'purpose=furniture/equipment',
  'op': '<=',
  'thr': 0.18370826542377472,
  'is_continuous': True},
 {'att': 'foreign_worker=no',
  'op': '<=',
  'thr': 0.7168410122394562,
  'is_continuous': True},
 {'att': 'purpose=domestic appliances',
  'op': '<=',
  'thr': 1.015466570854187,
  'is_continuous': True},
 {'att': 'savings=.. >= 1000 DM ',
  'op': '<=',
  'thr': 0.7176859378814697,
  'is_continuous': True},
 {'att': 'purpose=(vacation - does not exist?)',
  'op': '<=',
  'thr': 0.4622504562139511,
  'is_continuous': True},
 {'att': 'credit_history=critical account/ other credits existing (not at this bank)',
 

In [51]:
 for r in rules:
    print(r['att'])

age
credit_amount
purpose=retraining
duration_in_month
purpose=furniture/equipment
foreign_worker=no
purpose=domestic appliances
savings=.. >= 1000 DM 
purpose=(vacation - does not exist?)
credit_history=critical account/ other credits existing (not at this bank)
people_under_maintenance


In [52]:
df_range=pd.concat({'min':X_train.min(), 'max':X_train.max()},axis=1)

In [53]:
df_range=df_range.reset_index()

In [54]:
df_range

,index,min,max
0,duration_in_month,4,60
1,credit_amount,338,15945
2,installment_as_income_perc,1,4
3,present_res_since,1,4
4,age,19,75
...,...,...,...
56,job=unskilled - resident,0,1
57,telephone=none,0,1
58,"telephone=yes, registered under the customers ...",0,1
59,foreign_worker=no,0,1


In [55]:
rules

[{'att': 'age', 'op': '<=', 'thr': 20.726173400878906, 'is_continuous': True},
 {'att': 'credit_amount',
  'op': '>',
  'thr': -439.6443485021591,
  'is_continuous': True},
 {'att': 'purpose=retraining',
  'op': '<=',
  'thr': 0.11524588242173195,
  'is_continuous': True},
 {'att': 'duration_in_month',
  'op': '>',
  'thr': -1.9407005310058594,
  'is_continuous': True},
 {'att': 'purpose=furniture/equipment',
  'op': '<=',
  'thr': 0.18370826542377472,
  'is_continuous': True},
 {'att': 'foreign_worker=no',
  'op': '<=',
  'thr': 0.7168410122394562,
  'is_continuous': True},
 {'att': 'purpose=domestic appliances',
  'op': '<=',
  'thr': 1.015466570854187,
  'is_continuous': True},
 {'att': 'savings=.. >= 1000 DM ',
  'op': '<=',
  'thr': 0.7176859378814697,
  'is_continuous': True},
 {'att': 'purpose=(vacation - does not exist?)',
  'op': '<=',
  'thr': 0.4622504562139511,
  'is_continuous': True},
 {'att': 'credit_history=critical account/ other credits existing (not at this bank)',
 

In [56]:
df_rules = pd.DataFrame.from_records(rules)
df_rules

,att,op,thr,is_continuous
0,age,<=,20.726173,True
1,credit_amount,>,-439.644349,True
2,purpose=retraining,<=,0.115246,True
3,duration_in_month,>,-1.940701,True
4,purpose=furniture/equipment,<=,0.183708,True
5,foreign_worker=no,<=,0.716841,True
6,purpose=domestic appliances,<=,1.015467,True
7,savings=.. >= 1000 DM,<=,0.717686,True
8,purpose=(vacation - does not exist?),<=,0.462250,True
9,credit_history=critical account/ other credits...,<=,0.908596,True


In [57]:
df_viz = df_range.merge(df_rules,how='left',left_on='index',right_on='att')
df_viz = df_viz.drop('att', axis=1)
df_viz

,index,min,max,op,thr,is_continuous
0,duration_in_month,4,60,>,-1.940701,True
1,credit_amount,338,15945,>,-439.644349,True
2,installment_as_income_perc,1,4,NaN,NaN,NaN
3,present_res_since,1,4,NaN,NaN,NaN
4,age,19,75,<=,20.726173,True
...,...,...,...,...,...,...
56,job=unskilled - resident,0,1,NaN,NaN,NaN
57,telephone=none,0,1,NaN,NaN,NaN
58,"telephone=yes, registered under the customers ...",0,1,NaN,NaN,NaN
59,foreign_worker=no,0,1,<=,0.716841,True


In [58]:
df_viz['inst'] = real_inst.tolist()
df_viz

,index,min,max,op,thr,is_continuous,inst
0,duration_in_month,4,60,>,-1.940701,True,12
1,credit_amount,338,15945,>,-439.644349,True,1295
2,installment_as_income_perc,1,4,NaN,NaN,NaN,3
3,present_res_since,1,4,NaN,NaN,NaN,1
4,age,19,75,<=,20.726173,True,25
...,...,...,...,...,...,...,...
56,job=unskilled - resident,0,1,NaN,NaN,NaN,0
57,telephone=none,0,1,NaN,NaN,NaN,1
58,"telephone=yes, registered under the customers ...",0,1,NaN,NaN,NaN,0
59,foreign_worker=no,0,1,<=,0.716841,True,0


In [59]:
thr2_list=[]
for i, row in df_viz.iterrows():
    if (row['op']=='>' or row['op']== '>='):
        thr2_list.append(row['max'])
        continue
    if (row['op']=='<' or row['op']== '<='):
        thr2_list.append(row['min'])
        continue
    else:
        thr2_list.append(np.nan)
df_viz['thr2'] = thr2_list
df_viz

,index,min,max,op,thr,is_continuous,inst,thr2
0,duration_in_month,4,60,>,-1.940701,True,12,60.0
1,credit_amount,338,15945,>,-439.644349,True,1295,15945.0
2,installment_as_income_perc,1,4,NaN,NaN,NaN,3,NaN
3,present_res_since,1,4,NaN,NaN,NaN,1,NaN
4,age,19,75,<=,20.726173,True,25,19.0
...,...,...,...,...,...,...,...,...
56,job=unskilled - resident,0,1,NaN,NaN,NaN,0,NaN
57,telephone=none,0,1,NaN,NaN,NaN,1,NaN
58,"telephone=yes, registered under the customers ...",0,1,NaN,NaN,NaN,0,NaN
59,foreign_worker=no,0,1,<=,0.716841,True,0,0.0


In [62]:
features=df_viz['index'].to_list()
features

['duration_in_month',
 'credit_amount',
 'installment_as_income_perc',
 'present_res_since',
 'age',
 'credits_this_bank',
 'people_under_maintenance',
 'account_check_status=0 <= ... < 200 DM',
 'account_check_status=< 0 DM',
 'account_check_status=>= 200 DM / salary assignments for at least 1 year',
 'account_check_status=no checking account',
 'credit_history=all credits at this bank paid back duly',
 'credit_history=critical account/ other credits existing (not at this bank)',
 'credit_history=delay in paying off in the past',
 'credit_history=existing credits paid back duly till now',
 'credit_history=no credits taken/ all credits paid back duly',
 'purpose=(vacation - does not exist?)',
 'purpose=business',
 'purpose=car (new)',
 'purpose=car (used)',
 'purpose=domestic appliances',
 'purpose=education',
 'purpose=furniture/equipment',
 'purpose=radio/television',
 'purpose=repairs',
 'purpose=retraining',
 'savings=.. >= 1000 DM ',
 'savings=... < 100 DM',
 'savings=100 <= ... <

In [60]:
ch_list=[]
for i, row in df_viz.iterrows():
    if row['inst']!=0 or row['is_continuous']==True:
        p=alt.Chart(
            df_viz[df_viz['index']==row['index']]
        ).mark_rule(
            color='red' if row['is_continuous'] == True else'black',
            size=2
        ).encode(
            x=alt.X(
                field='inst',
                type='quantitative',
                title=None,
                scale= alt.Scale(domain=(row['min'],row['max']),clamp=True,nice=False)
            ),
            tooltip=[alt.Tooltip(field='inst',title=row['index'])]
        )
        
        t_min = alt.Chart(
            df_viz[df_viz['index']==row['index']]
        ).mark_text(
            color='black',
            dx=-10,
            align='right',
#             fontWeight='bold'
        ).encode(
            x=alt.X(
                field='min',
                type='quantitative',
                title=None
            ),
            text='min:N'
        )

        t_max = alt.Chart(
            df_viz[df_viz['index']==row['index']]
        ).mark_text(
            color='grey',
            dx=5,
            align='left',
            fontWeight='bold'
        ).encode(
            x=alt.X(
                field='max',
                type='quantitative',
                title=None
            ),
            text='max:N'
        )
            
        b =alt.Chart(
            df_viz[df_viz['index']==row['index']]
        ).mark_bar(
            color='#f4dd4d',size=5
        ).encode(
            x=alt.X(
                field='thr',
                type='quantitative',
                title=None,
            ),
            x2='thr2',
            y=alt.Y(field='index',type='nominal',title=None),

        )
        
        l =alt.Chart(
            df_viz[df_viz['index']==row['index']]
        ).mark_bar(
            color='grey',size=1
        ).encode(
            x=alt.X(
                field='min',
                type='quantitative',
                title=None,
                scale= alt.Scale(domain=(row['min'],row['max']),clamp=True,nice=False)
            ),
            x2='max',
            y=alt.Y(field='index',type='nominal',title=None),
        )
        
        
        comp = alt.layer(l,b,t_min,t_max,p).properties(
            height=10,
            width=100
        )
        ch_list.append(comp)
concat=alt.vconcat(*ch_list,  title=f"Predicted class: {exp.expDict['bb_pred']}")

concat.configure_concat(
    spacing=0
).configure_axis(
    grid=False
).configure_view(
    strokeWidth=1
).configure_axisX(
    disable=True
).configure_axisY(
    domain=False,
    ticks=False,
    labelPadding=50,
    minExtent=300
).configure_title(
    fontWeight='bold', anchor="start"
)

alt.VConcatChart(...)

## To Do

- Riscrivere il codice in una funzione ammodino
- Check su cosa prendere per le regole e le soglie
- Usare la FI per ordinare le feature 
- hconcat con la FI (plot accanto a plot)
- aggiungere il cutoff su entrambi
- inserire progressive disclosure:
    - filtrare per feature a cui è associata Rules
    - filtrare per FI (cutoff)
- Inserire nella funzione per plottare il preprocessing dei dati 

- Investigare le CR
- Fare la stessa cosa con titanic

- nel paper partire dalla vecchia viz html (linguaggio naturale)

In [ ]:
#da rivedere, generata male con chat gpt
def plot_viz(df_viz, exp):
    ch_list=[]
    for i, row in df_viz.iterrows():
        if row['inst']!=0 or row['is_continuous']==True:
            p=alt.Chart(
                df_viz[df_viz['index']==row['index']]
            ).mark_rule(
                color='red' if row['is_continuous'] == True else'black',
                size=2
            ).encode(
                x=alt.X(
                    field='inst',
                    type='quantitative',
                    title=None,
                    scale= alt.Scale(domain=(row['min'],row['max']),clamp=True,nice=False)
                ),
                tooltip=[alt.Tooltip(field='inst',title=row['index'])]
            )

            t_min = alt.Chart(
                df_viz[df_viz['index']==row['index']]
            ).mark_text(
                color='black',
                dx=-10,
                align='right',
            ).encode(
                x=alt.X(
                    field='min',
                    type='quantitative',
                    title=None
                ),
                text='min:N'
            )

            t_max = alt.Chart(
                df_viz[df_viz['index']==row['index']]
            ).mark_text(
                color='grey',
                dx=5,
                align='left',
                fontWeight='bold'
            ).encode(
                x=alt.X(
                    field='max',
                    type='quantitative',
                    title=None
                ),
                text='max:N'
            )

            b =alt.Chart(
                df_viz[df_viz['index']==row['index']]
            ).mark_bar(
                color='#f4dd4d',size=5
            ).encode(
                x=alt.X(
                    field='thr',
                    type='quantitative',
                    title=None,
                ),
                x2='thr2',
                y=alt.Y(field='index',type='nominal',title=None),

            )

            l =alt.Chart(
                df_viz[df_viz['index']==row['index']]
            ).mark_bar(
                color='grey',size=1
            ).encode(
                x=alt.X(
                    field='min',
                    type='quantitative',
                    title=None,
                    scale= alt.Scale(domain=(row['min'],row['max']),clamp=True,nice=False)
                ),
                x2='max',
                y=alt.Y(field='index',type='nominal',title=None),
            )


            comp = alt.layer(l,b,t_min,t_max,p).properties(
                height=10,
                width=100
            )
            ch_list.append(comp)
    concat=alt.vconcat(*ch_list,  title=f"Pred
